# Comparative Analysis of Snow Cover Metrics and Surface Energy Fluxes
## Northern Areas of Pakistan (2015-2024)

---


In [ ]:
# Import required libraries
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from shapely.geometry import mapping
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

# Spatial analysis libraries
from esda.moran import Moran, Moran_Local
from libpysal.weights import Queen, KNN
from splot.esda import moran_scatterplot
from spreg import OLS
from mapclassify import EqualInterval, NaturalBreaks, Quantiles

# Interactive mapping
import folium
from folium import Choropleth

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Define variables and their descriptions
variables_01 = ['asn', 'snowc', 'rsn', 'sde', 'sd', 'sf', 'smlt', 'tsn', 'fal',
                'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'es']

variables_02 = ['Snow Albedo', 'Snow Cover', 'Snow Density', 'Snow Depth', 
                'Snow Depth Water Equiv', 'Snowfall', 'Snowmelt', 
                'Temp of Snow Layer', 'Forecast Albedo',
                'Surface Upward Latent Heat Flux',
                'Surface Net Shortwave Solar Radiation',
                'Surface Net Longwave Thermal Radiation',
                'Surface Upward Sensible Heat Flux',
                'Surface Shortwave Solar Radiation Downwards',
                'Surface Longwave Thermal Radiation Downwards',
                'Snow Evaporation']

variables = dict(zip(variables_01, variables_02))

# Display variable mapping
pd.DataFrame(list(variables.items()), columns=['Code', 'Description'])

---
## 2. Spatial Analysis: District-Level Processing

In [ ]:
# Process each district polygon
polygon_data = {var: [] for var in variables}
years = None

print("Processing districts...")
for idx, polygon in gt_polygons.geometry.items():
    # Clip dataset to current polygon
    clipped = dataset.rio.clip([mapping(polygon)], gt_polygons.crs, drop=True, invert=False)
    
    # Group by year and calculate mean
    ds_yearly = clipped.groupby('valid_time.year').mean('valid_time')
    
    if years is None:
        years = ds_yearly.year.values[1:]  # diff reduces length by 1
    
    # Compute yearly difference and spatial average
    for var in variables:
        if var in ds_yearly:
            diff = ds_yearly[var].diff('year')
            spatial_mean = diff.mean(dim=['latitude', 'longitude'], skipna=True).values
            polygon_data[var].append(spatial_mean)
        else:
            polygon_data[var].append([float('nan')] * len(years))
    
    if (idx + 1) % 5 == 0:
        print(f"Processed {idx + 1}/{len(gt_polygons)} districts")

print("✓ District processing complete")

In [ ]:
# Create dataframe with yearly differences
all_columns = {}

for var in variables:
    data_array = pd.DataFrame(
        polygon_data[var],
        columns=[f"{var}_diff_{year}" for year in years],
        index=gt_polygons.index
    )
    for col in data_array.columns:
        all_columns[col] = data_array[col]

diff_df = pd.DataFrame(all_columns, index=gt_polygons.index)
gt_polygons = pd.concat([gt_polygons, diff_df], axis=1)

print(f"Added {len(diff_df.columns)} temporal difference columns")
gt_polygons.head()

---
## 3. Time Series Analysis with Enhanced Visualizations

In [ ]:
# Create comprehensive time series analysis for key variables
key_vars = ['snowc', 'sf', 'smlt', 'slhf', 'ssr']
var_labels = {
    'snowc': 'Snow Cover (%)',
    'sf': 'Snowfall (m × 10⁴)',
    'smlt': 'Snowmelt (m × 10⁴)',
    'slhf': 'Latent Heat Flux (J/m²)',
    'ssr': 'Net Solar Radiation (J/m²)'
}

# Prepare data for each variable
time_series_data = {}

for var in key_vars:
    var_years = sorted([int(col.split('_')[-1]) for col in gt_polygons.columns 
                        if col.startswith(f"{var}_diff_") and col.split('_')[-1].isdigit()])
    
    # Calculate statistics
    mean_changes = [gt_polygons[f"{var}_diff_{year}"].mean(skipna=True) for year in var_years]
    std_changes = [gt_polygons[f"{var}_diff_{year}"].std(skipna=True) for year in var_years]
    
    # Apply scaling for sf and smlt
    scale = 10000 if var in ['sf', 'smlt'] else 1
    mean_changes = [x * scale for x in mean_changes]
    std_changes = [x * scale for x in std_changes]
    
    time_series_data[var] = {
        'years': var_years,
        'mean': mean_changes,
        'std': std_changes,
        'upper': [m + s for m, s in zip(mean_changes, std_changes)],
        'lower': [m - s for m, s in zip(mean_changes, std_changes)]
    }

print("✓ Time series data prepared for all key variables")

In [ ]:
# Create interactive Plotly time series with uncertainty bands
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[var_labels[var] for var in key_vars] + [''],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
positions = [(1,1), (1,2), (2,1), (2,2), (3,1)]

for idx, var in enumerate(key_vars):
    data = time_series_data[var]
    row, col = positions[idx]
    color = colors[idx]
    
    # Add uncertainty band
    fig.add_trace(
        go.Scatter(
            x=data['years'] + data['years'][::-1],
            y=data['upper'] + data['lower'][::-1],
            fill='toself',
            fillcolor=color,
            opacity=0.2,
            line=dict(color='rgba(255,255,255,0)'),
            showlegend=False,
            hoverinfo='skip',
            name='±1 SD'
        ),
        row=row, col=col
    )
    
    # Add mean line
    fig.add_trace(
        go.Scatter(
            x=data['years'],
            y=data['mean'],
            mode='lines+markers',
            name=var.upper(),
            line=dict(color=color, width=2.5),
            marker=dict(size=6),
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(
    title_text="<b>Annual Average Changes in Snow Cover and Energy Flux Parameters (2015-2024)</b>",
    title_x=0.5,
    title_font_size=16,
    height=900,
    hovermode='x unified',
    template='plotly_white',
    font=dict(size=11)
)

# Update axes
fig.update_xaxes(title_text="Year", showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(title_text="Change", showgrid=True, gridwidth=1, gridcolor='lightgray')

fig.show()

In [ ]:
# Create climate scenario-style projection plot (similar to uploaded image)
def create_scenario_plot(var, title, ylabel, scenarios=None):
    """
    Creates a climate projection style plot with historical data and future scenarios
    """
    data = time_series_data[var]
    
    fig = go.Figure()
    
    # Split data into historical and projection periods
    split_year = 2020
    hist_idx = [i for i, y in enumerate(data['years']) if y < split_year]
    proj_idx = [i for i, y in enumerate(data['years']) if y >= split_year]
    
    # Historical period (blue)
    if hist_idx:
        hist_years = [data['years'][i] for i in hist_idx]
        hist_mean = [data['mean'][i] for i in hist_idx]
        hist_upper = [data['upper'][i] for i in hist_idx]
        hist_lower = [data['lower'][i] for i in hist_idx]
        
        # Uncertainty band
        fig.add_trace(go.Scatter(
            x=hist_years + hist_years[::-1],
            y=hist_upper + hist_lower[::-1],
            fill='toself',
            fillcolor='rgba(68, 114, 196, 0.3)',
            line=dict(color='rgba(255,255,255,0)'),
            showlegend=False,
            hoverinfo='skip'
        ))
        
        # Mean line
        fig.add_trace(go.Scatter(
            x=hist_years,
            y=hist_mean,
            mode='lines',
            name='Historical',
            line=dict(color='#4472C4', width=3)
        ))
    
    # Projection period (green/red for different scenarios)
    if proj_idx:
        proj_years = [data['years'][i] for i in proj_idx]
        proj_mean = [data['mean'][i] for i in proj_idx]
        proj_upper = [data['upper'][i] for i in proj_idx]
        proj_lower = [data['lower'][i] for i in proj_idx]
        
        # Moderate scenario (green)
        fig.add_trace(go.Scatter(
            x=proj_years + proj_years[::-1],
            y=proj_upper + proj_lower[::-1],
            fill='toself',
            fillcolor='rgba(112, 173, 71, 0.3)',
            line=dict(color='rgba(255,255,255,0)'),
            showlegend=False,
            hoverinfo='skip'
        ))
        
        fig.add_trace(go.Scatter(
            x=proj_years,
            y=proj_mean,
            mode='lines',
            name='Observed Trend (2020-2025)',
            line=dict(color='#70AD47', width=3)
        ))
    
    # Update layout to match climate projection style
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.5,
            xanchor='center',
            font=dict(size=14)
        ),
        xaxis=dict(
            title="Year",
            showgrid=True,
            gridcolor='lightgray',
            range=[min(data['years'])-1, max(data['years'])+1]
        ),
        yaxis=dict(
            title=ylabel,
            showgrid=True,
            gridcolor='lightgray'
        ),
        plot_bgcolor='white',
        height=400,
        hovermode='x unified',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
            bgcolor="rgba(255,255,255,0.8)"
        )
    )
    
    return fig

# Create plots for key variables
fig1 = create_scenario_plot('snowc', 
                            'Projection of Annual Average Snow Cover', 
                            'Snow Cover (%)')
fig1.show()

fig2 = create_scenario_plot('smlt', 
                            'Projection of Annual Average Snowmelt', 
                            'Snowmelt (m × 10⁴)')
fig2.show()

---
## 4. Spatial Visualization: Choropleth Maps

In [ ]:
# Animated Choropleth: Year-by-Year Evolution
gt_wgs84 = gt_polygons.to_crs(epsg=4326)

# Prepare animation data
animation_data = []
for year in years:
    col = f'smlt_diff_{year}'
    if col in gt_wgs84.columns:
        temp_df = gt_wgs84[['DISTRICT', 'geometry', col]].copy()
        temp_df['year'] = year
        temp_df['value'] = temp_df[col] * 10000
        animation_data.append(temp_df)

anim_df = pd.concat(animation_data, ignore_index=True)

fig = px.choropleth(
    anim_df,
    geojson=anim_df.geometry,
    locations=anim_df.index,
    color='value',
    animation_frame='year',
    hover_name='DISTRICT',
    color_continuous_scale='RdYlBu',
    range_color=[anim_df['value'].quantile(0.05), anim_df['value'].quantile(0.95)],
    title="<b>Snowmelt Evolution Across Northern Pakistan (2016-2025)</b>"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    height=700,
    coloraxis_colorbar=dict(title="Snowmelt<br>(m × 10⁴)"),
    updatemenus=[dict(
        type="buttons",
        buttons=[dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 800}}])]
    )]
)

fig.show()

In [ ]:
# Funnel Chart: Ranked District Impact
funnel_data = gt_polygons[['DISTRICT', 'smlt_diff_2024']].dropna()
funnel_data = funnel_data.sort_values('smlt_diff_2024', ascending=False).head(10)
funnel_data['smlt_diff_2024'] *= 10000

fig = go.Figure(go.Funnel(
    y=funnel_data['DISTRICT'],
    x=funnel_data['smlt_diff_2024'],
    textposition="inside",
    textinfo="value+percent initial",
    marker={"color": ["deepskyblue", "lightsalmon", "tan", "teal", "silver",
                     "darkorange", "lightgreen", "coral", "gold", "orchid"]}
))

fig.update_layout(
    title="<b>Top 10 Districts: Snowmelt Impact Funnel (2024)</b>",
    height=600
)

fig.show()

In [ ]:
# Create period aggregations
selected_variables = ['snowc', 'sf', 'smlt', 'slhf']
year_groups = {
    '2015–2019': list(range(2015, 2020)),
    '2020–2024': list(range(2020, 2025))
}

scaling_factors = {'sf': 10000, 'smlt': 10000, 'snowc': 1, 'slhf': 1}
classification_methods = {
    'sf': Quantiles,
    'smlt': NaturalBreaks,
    'snowc': EqualInterval,
    'slhf': EqualInterval
}

# Calculate period averages
for var in selected_variables:
    scale = scaling_factors.get(var, 1)
    for label, years_list in year_groups.items():
        agg_col_name = f"{var}_diff_{label.replace('–', '_')}"
        cols = [f"{var}_diff_{year}" for year in years_list if f"{var}_diff_{year}" in gt_polygons.columns]
        if cols:
            gt_polygons[agg_col_name] = gt_polygons[cols].mean(axis=1) * scale

print("✓ Period aggregations calculated")

In [ ]:
# Create static matplotlib choropleth comparison maps
cmap = 'RdYlBu'

for var in selected_variables:
    scale = scaling_factors.get(var, 1)
    colnames = [f"{var}_diff_{label.replace('–', '_')}" for label in year_groups]
    combined_vals = pd.concat([gt_polygons[col] for col in colnames if col in gt_polygons], axis=0).dropna()
    
    classifier_class = classification_methods.get(var, EqualInterval)
    try:
        classifier = classifier_class(combined_vals, k=5)
        bins = classifier.bins
    except Exception as e:
        print(f"Classification failed for {var}: {e}")
        continue
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    
    for i, label in enumerate(year_groups):
        agg_col_name = f"{var}_diff_{label.replace('–', '_')}"
        ax = axes[i]
        
        if agg_col_name in gt_polygons.columns:
            plot_data = gt_polygons.copy()
            
            plot_data.plot(
                column=agg_col_name,
                cmap=cmap,
                scheme='UserDefined',
                classification_kwds={'bins': bins},
                legend=True,
                legend_kwds={"fmt": "{:.2f}", "loc": "lower right"},
                ax=ax,
                edgecolor='black',
                linewidth=0.5,
                missing_kwds={"color": "lightgrey", "label": "Missing"}
            )
            
            # Add district labels
            for idx, row in plot_data.iterrows():
                if row['geometry'].area > 1e-4:
                    point = row['geometry'].representative_point()
                    txt = ax.text(
                        point.x, point.y,
                        s=row['DISTRICT'],
                        fontsize=7,
                        fontweight='bold',
                        ha='center',
                        va='center',
                        color='black',
                        zorder=10
                    )
                    txt.set_path_effects([
                        path_effects.Stroke(linewidth=2, foreground='white'),
                        path_effects.Normal()
                    ])
            
            ax.set_title(f"{variables[var]} ({label})\n(scaled ×{scale})", fontsize=12, fontweight='bold')
            ax.set_axis_off()
    
    plt.suptitle(f"{variables[var]}: Comparison 2015–2019 vs 2020–2024", 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.show()

In [ ]:
# Create interactive Plotly choropleth maps
gt_polygons_wgs84 = gt_polygons.to_crs(epsg=4326)

for var in selected_variables:
    for label in year_groups:
        agg_col_name = f"{var}_diff_{label.replace('–', '_')}"
        
        if agg_col_name in gt_polygons_wgs84.columns:
            fig = px.choropleth(
                gt_polygons_wgs84,
                geojson=gt_polygons_wgs84.geometry,
                locations=gt_polygons_wgs84.index,
                color=agg_col_name,
                hover_name='DISTRICT',
                hover_data={agg_col_name: ':.3f'},
                color_continuous_scale='RdYlBu',
                title=f"{variables[var]} - Average Change ({label})"
            )
            
            fig.update_geos(fitbounds="locations", visible=False)
            fig.update_layout(height=600, margin={"r":0,"t":50,"l":0,"b":0})
            fig.show()
            break  # Show only one period for brevity

---
## 5. Trend Change Analysis: Snowmelt

In [ ]:
# Calculate snowmelt trend change between periods
gt_polygons["smlt_trend_change"] = (
    gt_polygons["smlt_diff_2020_2024"] - gt_polygons["smlt_diff_2015_2019"]
)

# Static map
fig, ax = plt.subplots(1, 1, figsize=(12, 9))
gt_polygons.plot(
    column="smlt_trend_change",
    cmap="RdBu",
    linewidth=0.8,
    edgecolor="0.2",
    legend=True,
    ax=ax,
    scheme="quantiles",
    legend_kwds={"fmt": "{:.2e}", "loc": "lower right"}
)

for idx, row in gt_polygons.iterrows():
    point = row['geometry'].representative_point()
    txt = ax.text(point.x, point.y, s=row['DISTRICT'], 
                 fontsize=8, fontweight='bold', ha='center', color='black')
    txt.set_path_effects([path_effects.Stroke(linewidth=2, foreground='white'),
                          path_effects.Normal()])

ax.set_title("Change in Snowmelt Trend\n(2020–2024 vs 2015–2019)", 
            fontsize=14, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Interactive Folium map
gt_polygons_wgs84 = gt_polygons.to_crs(epsg=4326)

m = folium.Map(
    location=[gt_polygons_wgs84.geometry.centroid.y.mean(),
             gt_polygons_wgs84.geometry.centroid.x.mean()],
    zoom_start=6.5,
    tiles="CartoDB positron"
)

Choropleth(
    geo_data=gt_polygons_wgs84,
    name='Snowmelt Trend Change',
    data=gt_polygons_wgs84,
    columns=[gt_polygons_wgs84.index, 'smlt_trend_change'],
    key_on='feature.id',
    fill_color='RdBu',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Snowmelt Trend Change (2020–2024 vs 2015–2019)',
).add_to(m)

# Add district labels
for idx, row in gt_polygons_wgs84.iterrows():
    folium.Marker(
        location=[row.geometry.centroid.y, row.geometry.centroid.x],
        tooltip=f"{row['DISTRICT']}: {row['smlt_trend_change']:.2e}",
        icon=folium.DivIcon(html=f"<div style='font-size: 9pt; font-weight: bold;'>{row['DISTRICT']}</div>")
    ).add_to(m)

folium.LayerControl().add_to(m)
m

---
## 6. Long-term Percentage Change: Snow Depth

In [ ]:
# Calculate 10-year percentage change in snow depth
sde_cols = [col for col in gt_polygons.columns if col.startswith('sde_diff_')]
sde_diffs = gt_polygons[sde_cols]

# Calculate percentage change
gt_polygons['sde_pct_change'] = (sde_diffs.sum(axis=1) / sde_diffs.iloc[:, 0].abs()) * 100

# Plot
plot_data = gt_polygons.dropna(subset=['sde_pct_change'])

fig, ax = plt.subplots(1, 1, figsize=(12, 9))
plot_data.plot(
    column='sde_pct_change',
    cmap='RdYlBu',
    scheme='NaturalBreaks',
    k=5,
    legend=True,
    edgecolor='black',
    linewidth=0.6,
    ax=ax,
    legend_kwds={"fmt": "{:.1f}%", "loc": "lower right"}
)

for idx, row in plot_data.iterrows():
    point = row['geometry'].representative_point()
    txt = ax.text(point.x, point.y, s=row['DISTRICT'], 
                 fontsize=8, fontweight='bold', ha='center', color='black')
    txt.set_path_effects([path_effects.Stroke(linewidth=2, foreground='white'),
                          path_effects.Normal()])

ax.set_title("Decadal Percentage Change in Snow Depth (2015-2024)", 
            fontsize=14, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 7. Spatial Autocorrelation Analysis

In [ ]:
# Moran's I for Snowfall (2020-2024)
column = 'sf_diff_2020_2024'
gdf = gt_polygons[[column, 'geometry']].dropna()

# Create spatial weights
w = KNN.from_dataframe(gdf, k=4)
w.transform = 'r'

x = gdf[column].values
moran_sf = Moran(x, w)

print("=" * 60)
print(f"GLOBAL MORAN'S I - {variables['sf']} (2020-2024)")
print("=" * 60)
print(f"Moran's I:          {moran_sf.I:.4f}")
print(f"p-value:            {moran_sf.p_sim:.4f}")
print(f"Expected I:         {moran_sf.EI:.4f}")
print(f"Interpretation:     ", end="")
if moran_sf.p_sim < 0.05:
    if moran_sf.I > 0:
        print("Significant positive spatial autocorrelation (clustering)")
    else:
        print("Significant negative spatial autocorrelation (dispersion)")
else:
    print("No significant spatial autocorrelation (random pattern)")
print("=" * 60)

In [ ]:
# Moran's I for Snowmelt (2020-2024)
column = 'smlt_diff_2020_2024'
gdf = gt_polygons[[column, 'geometry']].dropna()

w = KNN.from_dataframe(gdf, k=4)
w.transform = 'r'

x = gdf[column].values
moran_smlt = Moran(x, w)

print("=" * 60)
print(f"GLOBAL MORAN'S I - {variables['smlt']} (2020-2024)")
print("=" * 60)
print(f"Moran's I:          {moran_smlt.I:.4f}")
print(f"p-value:            {moran_smlt.p_sim:.4f}")
print(f"Expected I:         {moran_smlt.EI:.4f}")
print(f"Interpretation:     ", end="")
if moran_smlt.p_sim < 0.05:
    if moran_smlt.I > 0:
        print("Significant positive spatial autocorrelation (clustering)")
    else:
        print("Significant negative spatial autocorrelation (dispersion)")
else:
    print("No significant spatial autocorrelation (random pattern)")
print("=" * 60)

In [ ]:
# Moran scatterplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

moran_scatterplot(moran_sf, ax=axes[0])
axes[0].set_title(f"Moran's I Scatterplot: {variables['sf']} (2020-2024)", 
                 fontsize=12, fontweight='bold')

moran_scatterplot(moran_smlt, ax=axes[1])
axes[1].set_title(f"Moran's I Scatterplot: {variables['smlt']} (2020-2024)", 
                 fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Local Indicators of Spatial Association (LISA)

In [ ]:
# Local Moran's I for Snowmelt (2018 as example year)
variable = 'smlt_diff_2018'
gt_polygons[variable] = gt_polygons[variable].fillna(0).astype('float64')
data = gt_polygons[variable].values

# Create spatial weights
w = Queen.from_dataframe(gt_polygons)
w.transform = 'r'

# Compute Local Moran's I
m_local = Moran_Local(data, w)

# Add results to GeoDataFrame
gt_polygons['LISA_cluster'] = m_local.q
gt_polygons['LISA_p'] = m_local.p_sim

# Plot LISA clusters
fig, ax = plt.subplots(1, 1, figsize=(12, 9))
gt_polygons.plot(
    column='LISA_cluster', 
    cmap='Set3', 
    legend=True, 
    edgecolor='black',
    linewidth=0.6,
    ax=ax,
    categorical=True,
    legend_kwds={'title': 'Cluster Type\n1=HH, 2=LH, 3=LL, 4=HL', 'loc': 'lower right'}
)

for idx, row in gt_polygons.iterrows():
    point = row['geometry'].representative_point()
    txt = ax.text(point.x, point.y, s=row['DISTRICT'], 
                 fontsize=8, fontweight='bold', ha='center', color='black')
    txt.set_path_effects([path_effects.Stroke(linewidth=2, foreground='white'),
                          path_effects.Normal()])

ax.set_title(f"Local Moran's I (LISA) Clusters\n{variables['smlt']} Difference (2018)", 
            fontsize=14, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

# Display significant clusters
significant = gt_polygons[gt_polygons['LISA_p'] < 0.05]
print(f"\nSignificant LISA clusters (p < 0.05): {len(significant)} districts")
print(significant[['DISTRICT', 'LISA_cluster', 'LISA_p']])

---
## 9. Spatial Regression Analysis

In [ ]:
# OLS Regression: Snowmelt ~ Latent Heat Flux + Solar Radiation + Albedo (2024)
X = gt_polygons[['slhf_diff_2024', 'ssr_diff_2024', 'fal_diff_2024']].fillna(0).values
y = gt_polygons['smlt_diff_2024'].fillna(0).values.reshape(-1, 1)

# Spatial weights
w = KNN.from_dataframe(gt_polygons, k=4)
w.transform = 'r'

# Fit OLS model
model = OLS(y, X, w=w, name_y='Snowmelt', name_x=['Latent Heat', 'Solar Radiation', 'Albedo'])

print("=" * 80)
print("SPATIAL REGRESSION: Snowmelt (2024)")
print("=" * 80)
print(model.summary)
print("=" * 80)

---
## 10. Correlation Analysis

In [ ]:
# Create correlation matrix for 2024 variables
corr_vars = ['snowc_diff_2024', 'sf_diff_2024', 'smlt_diff_2024', 
             'slhf_diff_2024', 'ssr_diff_2024', 'sde_diff_2024']

corr_data = gt_polygons[corr_vars].fillna(0)
corr_matrix = corr_data.corr()

# Rename for clarity
labels = ['Snow Cover', 'Snowfall', 'Snowmelt', 'Latent Heat', 'Solar Rad', 'Snow Depth']
corr_matrix.index = labels
corr_matrix.columns = labels

# Plotly heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=labels,
    y=labels,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont={"size": 11},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title="<b>Correlation Matrix: Snow and Energy Variables (2024)</b>",
    title_x=0.5,
    xaxis_title="",
    yaxis_title="",
    height=600,
    width=700
)

fig.show()

In [ ]:
# Scatter matrix for key relationships
scatter_data = gt_polygons[['smlt_diff_2024', 'slhf_diff_2024', 
                            'ssr_diff_2024', 'DISTRICT']].dropna()

fig = px.scatter_matrix(
    scatter_data,
    dimensions=['smlt_diff_2024', 'slhf_diff_2024', 'ssr_diff_2024'],
    color='DISTRICT',
    title="<b>Pairwise Relationships: Snowmelt, Latent Heat, and Solar Radiation (2024)</b>",
    labels={
        'smlt_diff_2024': 'Snowmelt',
        'slhf_diff_2024': 'Latent Heat',
        'ssr_diff_2024': 'Solar Radiation'
    },
    height=800
)

fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.show()

---
## 11. Summary Statistics and Key Findings

In [ ]:
# Summary statistics table
summary_vars = ['snowc_diff_2020_2024', 'sf_diff_2020_2024', 
                'smlt_diff_2020_2024', 'slhf_diff_2020_2024']

summary_stats = gt_polygons[summary_vars].describe().T
summary_stats['Variable'] = [variables[v.split('_')[0]] for v in summary_vars]
summary_stats = summary_stats[['Variable', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

print("\n" + "="*100)
print("SUMMARY STATISTICS: 2020-2024 Period")
print("="*100)
print(summary_stats.to_string())
print("="*100)

In [ ]:
# District rankings for snowmelt change
rankings = gt_polygons[['DISTRICT', 'smlt_diff_2020_2024']].dropna()
rankings = rankings.sort_values('smlt_diff_2020_2024', ascending=False)
rankings['Rank'] = range(1, len(rankings) + 1)
rankings = rankings[['Rank', 'DISTRICT', 'smlt_diff_2020_2024']]
rankings.columns = ['Rank', 'District', 'Avg Snowmelt Change (2020-2024)']

# Plotly bar chart
fig = go.Figure(data=[
    go.Bar(
        x=rankings['District'],
        y=rankings['Avg Snowmelt Change (2020-2024)'],
        marker_color=np.where(rankings['Avg Snowmelt Change (2020-2024)'] > 0, 
                              '#d62728', '#2ca02c'),
        text=rankings['Avg Snowmelt Change (2020-2024)'].round(2),
        textposition='outside'
    )
])

fig.update_layout(
    title="<b>District Rankings by Average Snowmelt Change (2020-2024)</b>",
    title_x=0.5,
    xaxis_title="District",
    yaxis_title="Snowmelt Change (m × 10⁴)",
    height=500,
    template='plotly_white',
    showlegend=False
)

fig.update_xaxes(tickangle=45)
fig.show()

print("\n" + "="*80)
print("TOP 5 DISTRICTS WITH HIGHEST SNOWMELT INCREASE")
print("="*80)
print(rankings.head().to_string(index=False))
print("\n" + "="*80)
print("TOP 5 DISTRICTS WITH HIGHEST SNOWMELT DECREASE")
print("="*80)
print(rankings.tail().to_string(index=False))
print("="*80)